# 03 · Baseline models (5-fold CV on train; hold-out test reported once)

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from IPython.display import Image, display, Markdown
from src import config
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)
M, F = config.METRICS_DIR, config.FIG_DIR
show = lambda name, w=900: display(Image(filename=str(F / name), width=w))

In [2]:
import matplotlib.pyplot as plt
from src.plotting import BLUE, ORANGE
for t in ('total', 'per_ha'):
    cv = pd.read_csv(M/f'cv_results_{t}.csv')
    display(Markdown(f'### Target: `{t}`'))
    display(cv[['model','stage','cv_R2','cv_R2_std','cv_RMSE','cv_MAE','cv_MAPE_%']].round(4))

### Target: `total`

,model,stage,cv_R2,cv_R2_std,cv_RMSE,cv_MAE,cv_MAPE_%
0,Mean predictor (reference),baseline,-0.0006,0.0008,9257.1517,8034.2954,57.9901
1,Hectares-only OLS (reference),baseline,0.9888,0.0012,977.9200,739.4826,3.6892
2,Linear Regression,baseline,0.9888,0.0012,976.9084,742.7557,3.7292
3,Random Forest,baseline,0.9909,0.0013,880.5404,639.3548,2.8604
4,Extra Trees,baseline,0.9904,0.0011,904.8526,653.6145,2.9468
5,Gradient Boosting,baseline,0.9917,0.0012,839.9168,611.2848,2.7337
6,LightGBM,baseline,0.9911,0.0012,873.8117,638.6585,2.8543
7,XGBoost,baseline,0.9906,0.0013,896.4548,652.4566,2.9229
8,LightGBM (Optuna),optuna,0.9919,0.0013,830.4915,600.7511,2.6793
9,XGBoost (Optuna),optuna,0.9918,0.0013,836.7690,605.9629,2.7100


### Target: `per_ha`

,model,stage,cv_R2,cv_R2_std,cv_RMSE,cv_MAE,cv_MAPE_%
0,Mean predictor (reference),baseline,-0.0007,0.0008,287.6900,249.2312,4.1931
1,Hectares-only OLS (reference),baseline,0.3514,0.0415,231.4208,183.0221,3.0924
2,Linear Regression,baseline,0.3482,0.0411,231.9997,183.8862,3.1071
3,Random Forest,baseline,0.4438,0.0638,214.0767,168.9110,2.8557
4,Extra Trees,baseline,0.4131,0.0609,219.9745,173.0032,2.9261
5,Gradient Boosting,baseline,0.5007,0.0533,202.8739,161.0174,2.7254
6,LightGBM,baseline,0.4501,0.0593,212.8858,168.5606,2.8492
7,XGBoost,baseline,0.4182,0.0596,219.0124,173.0529,2.9264
8,LightGBM (Optuna),optuna,0.5133,0.0581,200.2297,158.4294,2.6820
9,XGBoost (Optuna),optuna,0.5074,0.0585,201.4396,159.4027,2.6988


In [3]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, t in zip(axes, ('total', 'per_ha')):
    cv = pd.read_csv(M/f'cv_results_{t}.csv').query('stage == "baseline"').sort_values('cv_R2')
    ax.barh(cv.model, cv.cv_R2, xerr=cv.cv_R2_std, color=BLUE, height=0.6)
    ax.set(title=f'CV R² — {t}', xlabel='R²', xlim=(min(0, cv.cv_R2.min()), 1))
fig.tight_layout(); fig.savefig(F/'baseline_cv_r2.png', dpi=130)

## Findings

* **Total yield**: every model, including a one-variable OLS on hectares (CV R² 0.9888), scores R² ≈ 0.99. Tree ensembles reduce RMSE from ~978 kg to ~840 kg (≈14%) — the real gain of ML over the trivial baseline.
* **Per-hectare yield**: OLS ≈ 0.35, Random Forest 0.44, LightGBM 0.45, Gradient Boosting 0.50 (CV R²). Non-linear models help because the farm-size effect is a step (1–3 ha vs 4–6 ha) plus variety×size interactions.
